# Encoders
As mentioned in the pre-training notebook, **encoder models** are used for tasks where the goal is to make sense of the input data. So, tasks such as trying to decipher the meaning of a text or to understand the sentiment of text work are some examples of use cases for encoder models. The goal of rich contextual representation of text, then, will inform how a model is trained and the objective function that we use. Again, we will focus primarily on self-supervised modeling (as it's the most popular form), although encoders can be implemented with supervised paradigms.

## Pre-training
We saw in the pretraining notebook that the output of an encoder is expressed as
$$ H = g_{\hat{\theta}}(X) = \text{Encode}_{\hat{\theta}}(X) $$
That is, we produce a sequence of vectors $H=(h_0, h_1, ..., h_N)$. It turns out that training an encoder isn't easy and there's no gold standard way of doing so. A typical approach to encoder pre-training is to combine the encoder with some output layers to receive supervision signals that are easier to obtain. 

To do this, we stack a softmax layer on top of the encoder, which in turn outputs a sequence of probability distributions. 
$$
\begin{align}
    \text{Softmax}_W\left[\text{Encode}_{\hat{\theta}}(X)\right] &= \text{Softmax}_W\left[H\right]\\
    &= \text{Softmax}\left[H W\right]\\
        &= \begin{bmatrix}
           p^{W, \theta}_1 \\
           \vdots \\
           p^{W, \theta}_N
         \end{bmatrix}
\end{align}
$$
Here, the probability $p_i^{W, \theta}$ denotes the output distribution $\mathbb{P}(\cdot | x)$ of the $i$-th token. Additionally, we include learnable parameters $W$ for the supervised classification task. The reason we add attach a softmax classification head is so we can turn abstract vectors into a concrete prediction task that produces a loss. Intuitively, $p_i^{W, \theta}$ asks: "what word/token do I think belongs in position $i$?" So, we are essentially formulating the a **multi-class classification task over the entire vocabulary**. Now that we have predictions, we can compare them to the true token, which in turn gives us a way to calculate a loss function. 

It should be noted that the difference between the output distribution of encoders and decoders lies in the interpretation. For standard decoder language modeling, the distribution represents the possible next tokens in the sequence, whereas for encoders, **the entire sequence can be observed at once**. So, the probability of a word given both its left and right context. In practice, this objective is implemented by randomly masking tokens and training the model to reconstruct them from context, which we will see next.

### Masked language modeling (MLM)
**Masked language modeling** forms the basis of the well-known **BERT** model, first introduced by Devin et al. in 2019. The basic idea behind mask language modeling is to mask out certain tokens in the input with the [MASK] token, thereby creating openings for token predictions. One important caveat of MLM is the use of both left and right context (kind of like CBoW in word2vec). Since all of the unmasked tokens are used for word prediction, we have a **bidirectional** model.

More formally, given a input sequence $x_1, \cdots, x_N$, we have a set of masks $A(x) = \set{i_1, \cdots, i_u}$. We can denote the masked sequence as $\bar{x}$. For example, we might have something like
$$x = \text{The early bird catches the worm}$$
and the masked tokens $A(x)$ with 
$$\bar{x} = \text{The early [MASK] catches the [MASK]}$$

Now that we have the sequences $x$ and $\bar{x}$, we can perform the optimization step where we maximize the probabilities $p^{W,\theta}_i$ for each unmasked token $i\in A(x)$. Using MLE, we have the objective
$$(\hat{W}, \hat{\theta}) = \arg\max_{W, \theta} \sum_x \sum_{i\in A(x)} \log p_i^{W, \theta}(x_i | \bar{x})$$
where the first sum is over all possible sequences, and the second sum is over the masked tokens within a specific sequence. We then get the loss function

$$\text{Loss} = -\sum_x \sum_{i\in A(x)} \log p_i^{W, \theta}(x_i | \bar{x})$$

which can also be expressed as the cross-entropy loss

$$\text{Loss} = -\sum_x \sum_{i\in A(x)} y_{x_i} \log p_i^{W, \theta}(x_i | \bar{x})$$

where $y_{x_i}$ is the corresponding correct token for the $i$-th masked token in the sequence $x$.

Looking at our earlier example, 

$$
\begin{align*}
\text{Loss} &= \log \mathbb{P}(x_3 = \text{bird} | \bar{x} = \text{[CLS] The early [MASK] catches the [MASK]})\\
&\quad + \log \mathbb{P}(x_6 = \text{worm} | \bar{x} = \text{[CLS] The early [MASK] catches the [MASK]})
\end{align*}
$$

Once we have obtained the $\hat{W}$ and $\hat{\theta}$, we can then drop the $\hat{W}$ and perform fine-tuning for downstream tasks.

It should be noted that while MLM is the most popular approach, it does come with issues. For one, there's a discrepency between the inclusion of masked tokens during train vs. test time. That is, we don't include the masked tokens during evaluation. Furthermore, masked terms might be crucial for understanding the meaning of the sentence. For example, if we have a sequence like "Zendaya is a great actress," we can mask "Zendaya" and "Actress," which leads to the sentence losing all contextual meaning. One alternaive is permuted language modeling, which we won't go into detail here.

### Next sentence prediction (NSP)
Another innovation in the BERT model is the idea of **next sentence prediction**, where classification is performed over the sentence level. The base assumption of NSP is that a good classifier should be able to capture relationships not only between tokens within a sentence, but also between them. Thus, we want to be able to use the output of encoding two sentences $\text{sent}_A$ and $\text{sent}_B$, to determine whether $B$ comes after $A$. For example, take the sentence:

$$ \text{[CLS] Michael Jordan plays basketball. [SEP] He is the GOAT. [SEP]}$$

The way that NSP works largely follows the same steps as other encoder paradigms. Given input $x_0, x_1, \cdots, x_N$, we begin by obtaining outputs $h_0, h_1, ..., h_N$. The key distinction lies in the way we utilize the outputs. First note that $x_0$ is the embedding for the special token [CLS]. It turns out that $x_0$ tends to represent the sequence as a whole. The reason for this is not so much mathematical as it is more architectural. The [CLS] token, like all other tokens goes through the attention mechanism, absorbing context from other words. However, unlike other tokens, it has no lexical meaning. Thus, $h_0$ is quite literally a learned, attention-weighted aggregation of the entire sequence.

We can then use $h_0$ to stack a softmax layer on top of the encoder to construct a **binary classification** system. 
$$\mathbb{P}(y=1 | x) = \mathbb{P}(y=1 | \text{Sent}_A, \text{Sent}_B) = \sigma(h_0^\top W + b)$$
In order to generate training samples, we can utilize the natural sequence of two sentences in the text. For example, we obtain a positive sample by using actual consecutive sentences, and a negative sample by using randomly sampled sentences. In other words, We generate $\text{Sent}_A$/$\text{Sent}_B$ and negative examples from raw document order **before training**, and then force the model—via the NSP loss—to compress the notion of sentence-level coherence into the single vector.

For BERT, NSP is used as an additional training loss along with MLM. Other examples of binary classification systems are discriminant based ones (e.g. ELECTRA).

## BERT

Here, we look at the BERT model in a bit more details. As mentioned above, the loss in BERT is the combination of the MLM and NSP loss:
$$\text{Loss}_{\text{BERT}} = \text{Loss}_{\text{MLM}} + \text{Loss}_{\text{NSP}}$$
Given a input sequence of the form "[CLS] $\text{Sent}_A$. [SEP] $\text{Sent}_B$. [SEP]", we can obtain the MLM and NSP loss separately. For MLM, the first step is to randomly select a subset of tokens (15% in paper) to be altered. Of this subset,
- **Token masking**: 80% of tokens are masked.
- **Random replacement**: 10% of tokens are randomly replaced with random words. This is to make the model robust to noise. 
- **Unchanged**: 10% of tokens are unchanged. Note that we are still performing this (easy) prediction tasks. It's mostly used to guide the model to use easier evidence for prediction.

So, let $A(x)$ be the subset of altered tokens in the sequence. We obtain the modified sequence $\bar{x}$, which we can then use in the loss function:
$$\text{Loss}_{\text{MLM}} = \sum_x \sum_{i\in A(x)} \log p_i^{W, \theta}(x_i | \bar{x})$$

For NSP, each sequence $x_i$ is given label $c_i \in C=\set{\text{IsNext}, \text{NotNext}}$. For each sentence pair, the encoder gives us
$$p^{(i)}\left(c_i | \bar{x}_i = (\text{sent}_A^{(i)}, \text{sent}_B^{(i)})\right) = \sigma(h_i0^\top W + b)$$
We the obtain the overall log loss function across all training examples:
$$\text{Loss}_{\text{NSP}} = \sum_i \log p^{(i)}\left(c_i | \bar{x}_i\right)$$

As is regular in training deep neural networks, we optimize the model parameters by minimizing this loss. To do this, a number of training samples are collected. During training, a batch of training samples is randomly selected from this collection at a time, and LossBERT is accumulated over these training samples. Then, the model parameters are updated via gradient descent or its variants. This process is repeated many times until some stopping criterion is satisfied, such as when the training loss converges.

The transformer layers of BERT pretty much follows the canonical implementation, with one small caveat: the sequence of input embeddings have an additional **segment embedding** component $s_n$.
$$\tilde{x}_n = x_n + r_n + s_n$$
where $r_n$ is the position encoding. The segment embedding indicates whether a token belongs to $\text{Sent}_A$ or $\text{Sent}_B$.
![segment embeddings](segment_embedding.png)

## Fine-tuning
Now that we have a pre-trained BERT model, we can express the outputs as
$$H = g_{\hat{\theta}}(X) = \text{BERT}_{\hat{\theta}}(x)$$
However, from earlier discussions, we know that the model isn't quite ready for down-stream tasks. We will need additional fine tuning so that the model can adapt to these tasks. To do this, we need to incorporate a predictor $f_\omega(\cdot)$ (e.g. logistic regression, neural network, etc.). We can express this model as
$$ y = f_\omega(\text{BERT}_{\hat{\theta}}(x))$$

This step is usually supervised, where we have an additional fine-tuning training sample $\set{(x', y')}_m=1^M$ over which we can do an additional layer of training. The outcome is the optimized parameters $\tilde{\omega}$ and $\tilde{\theta}$ (or just $\tilde{\omega}$ holding $\hat{\theta}$ fixed). The form of the downstream tasks determines the input and output formats of the model, as well as the architecture of the predictor. In the following we list some tasks to which BERT models are generally suited.

- **Classification (single text)**: Here, BERT recieves an entire sequence of tokens and encodes it as a sequence of vectors. Typically, the first output $h_0$ (or $h_{\text{cls}})$ is used as the representation of the entire text. So, the predictor takes it as input for the downstream task. For example, we want to understand the sentiment of a tweet data. We fine-tune the model on labeled tweet data with corresponding sentiment information.
- **Classification (pairs of text)**: Classification can also be performed on a pair of texts. Suppose we have two texts, $x_1...x_m$ and $y_1...y_n$. We can concatenate these texts to form a single sequence with a length $\text{len}$. Then, we predict a label for this combined text sequence based on the $h_0$ vector. Text pair classification covers several problems, including semantic equivalence judgement (determine whether two texts are semantically equivalent), text entailment judgement (determine whether a hypothesis
can be logically inferred or entailed from a premise), grounded commonsense inference (determine whether an event is likely to happen given its context), and question-answering inference (determine whether an answer corresponds to a given question).
- **Regression**: Instead of generating a label distribution, we can have the prediction network output a real-valued score. For training or fine-tuning, we can minimize the regression loss of the model output as usual.
- **Sequence labeling**: Sequence labeling is a machine learning approach applicable to a wide range of NLP problems. This approach assigns a label to each token in an input sequence, and some linguistic annotations can then be derived from this sequence of labels. An example of sequence labeling in NLP is part-of-speech (POS) tagging, where each word in a sentence is labeled with its corresponding POS tag.

Look at the bottom of the pretraining notebook to see what hyperparameters needed for fine-tuning.